In [ ]:
# ==================================================
# Import libraries
# ==================================================
import os
import sys
import gc
import logging
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import optuna
import joblib
from datetime import datetime
from pathlib import Path
from collections import defaultdict
from importlib import reload

from sklearn.model_selection import StratifiedKFold
from torchmetrics.classification import (BinaryAccuracy, BinaryAUROC, BinaryF1Score, BinaryRecall, BinarySpecificity)
from captum.attr import IntegratedGradients

# ==================================================
# Function to print timestamps
# ==================================================

def log(msg):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}", flush=True)

# ==================================================
# Check GPU 
# ==================================================
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
print("GPU name:", torch.cuda.get_device_name(0))

# ==================================================
# Logging & determinism
# ==================================================
logging.basicConfig(filename="nested_cv_ResNet_log.txt", level=logging.INFO)

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#===========================================================================
# Parallelize repetitions and outer CV
# ==========================================================================
import argparse

parser = argparse.ArgumentParser()
parser.add_argument("--rep", type=int, required=True)
parser.add_argument("--fold", type=int, required=True)
args = parser.parse_args()

REP = args.rep
FOLD = args.fold


# ==================================================
# Paths & custom libs
# ==================================================
sys.path.append('/scratch/project_XX/AnalysisFiles')

from mylibs.load_datasets import spectral_dataloader, ConfoundRegressor
from mylibs.ResNet import ResNet
from mylibs.training import test_model

# ==================================================
# Seed
# ==================================================
def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed_all(seed)

# ==================================================
# Load data
# ==================================================
os.chdir('/scratch/project_XX/AnalysisFiles/Data_matrices_deep_learning')

XX = np.load('preprocessed_spectral_data_for_deep_learning.npy')
YY = np.load('class_labels_for_deep_learning.npy')
sample_id_vector = np.load('sample_id_vector_for_deep_learning.npy')

# ==================================================
# Load confounds
# ==================================================
os.chdir('/scratch/project_XX/AnalysisFiles/Data_matrices_averaged_samples')
clinical_data = pd.read_csv('cleaned_clinical_data.csv')
clinical_data_expanded = clinical_data.loc[clinical_data.index.repeat(450)].reset_index(drop=True)
confound_matrix = np.column_stack([clinical_data_expanded['AGE_AT_SAMPLING'].values, clinical_data_expanded['GENDER_BINARY'].values])


# ===========================================================================================
# Helper function to calculate baseline for IGs from the mean of the spectra in training data
# ===========================================================================================
def calculate_baseline(dataloader, device):
    all_sum = None
    n = 0

    for inputs, _, _ in dataloader:
        inputs = inputs.to(device)

        if all_sum is None:
            all_sum = inputs.sum(dim=0, keepdim=True)
        else:
            all_sum += inputs.sum(dim=0, keepdim=True)

        n += inputs.size(0)

    baseline = all_sum / n
    return baseline


# ==================================================
# Integrated Gradients
# ==================================================
def compute_integrated_gradients(model, testloader, trainloader, device, n_steps=100):
    model.eval()
    ig = IntegratedGradients(model)

    ig_by_class = defaultdict(list)
    delta_by_class = defaultdict(list)

    # Calculate baseline based on training data
    baseline = calculate_baseline(trainloader, device)

    # Follow spectral processing
    processed = 0
    total = len(testloader.dataset)

    for batch_idx, (x, y, _) in enumerate(testloader):

        x = x.to(device)
        y = y.to(device)

        for i in range(x.size(0)):

            processed += 1

            if processed % 100 == 0: 
                log(
                    f" IG progress: "
                    f"{processed}/{total} spectra"
                )

            xi = x[i:i+1]
            yi = y[i].item()

            attr, delta = ig.attribute(
                xi,
                target=yi,
                baselines=baseline,
                n_steps=n_steps,
                return_convergence_delta=True
            )

            ig_by_class[yi].append(attr.cpu().numpy())
            delta_by_class[yi].append(delta.cpu().numpy())

    for k in ig_by_class:
        ig_by_class[k] = np.concatenate(ig_by_class[k], axis=0)
        delta_by_class[k] = np.concatenate(delta_by_class[k], axis=0)

    return ig_by_class, delta_by_class

# ==================================================
# Optuna objective (INNER CV)
# ==================================================
def objective(trial, X, y, sample_ids, C):

    batch_size = trial.suggest_int("batch_size", 16, 128)
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    beta1 = trial.suggest_float("beta1", 0.8, 0.99)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-1, log=True)

    hidden_size = trial.suggest_int("hidden_size", 16, 256, log=True)
    num_layers  = trial.suggest_int("num_layers", 1, 5)
    block_size  = trial.suggest_int("block_size", 1, 5)

    unique_ids = np.unique(sample_ids)
    unique_labels = np.array([y[sample_ids == uid][0] for uid in unique_ids])

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=1234)

    patience = 3
    aucs = []
    fold_metrics = []

    for train, val in skf.split(unique_ids, unique_labels):

        train_ids = unique_ids[train]
        val_ids = unique_ids[val]

        train_mask = np.isin(sample_ids, train_ids)
        val_mask = np.isin(sample_ids, val_ids)

        conf = ConfoundRegressor()
        X_train, beta = conf.fit_transform(X[train_mask], C[train_mask])
        X_val = X[val_mask] - (C[val_mask] @ beta)

        trainloader = spectral_dataloader(X_train, y[train_mask], sample_ids[train_mask], batch_size=batch_size)

        valloader = spectral_dataloader(X_val, y[val_mask], sample_ids[val_mask], batch_size=batch_size, shuffle=False)

        model = ResNet([hidden_size]*num_layers, [block_size]*num_layers).to(device)

        opt = torch.optim.Adam(model.parameters(), lr=lr, betas=(beta1, 0.999), weight_decay=weight_decay)

        loss_fn = nn.CrossEntropyLoss()

        use_amp = device.type == "cuda"
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

        best_val_auc = 0
        patience_counter = 0

        # ============================
        # TRAIN + VALIDATE
        # ============================
        for epoch in range(30):

            # ----- TRAIN -----
            model.train()
            for xb, yb, _ in trainloader:
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)

                opt.zero_grad()

                with torch.cuda.amp.autocast(enabled=use_amp):
                    outputs = model(xb)
                    loss = loss_fn(outputs, yb)

                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()

            # ----- VALIDATE -----
            model.eval()
            probs, targs = [], []

            with torch.no_grad():
                for xb, yb, _ in valloader:
                    xb = xb.to(device, non_blocking=True)
                    batch_probs = torch.softmax(model(xb), dim=1)[:, 1].cpu()
                    probs.append(batch_probs)
                    targs.append(yb.cpu())

            probs_tensor = torch.cat(probs)
            targs_tensor = torch.cat(targs)

            preds_tensor = (probs_tensor > 0.5).long()

            val_auc = BinaryAUROC()(probs_tensor, targs_tensor).item()
            
            trial.report(val_auc, epoch) 
            if trial.should_prune():
                raise optuna.TrialPruned()

            # Early stopping
            if val_auc > best_val_auc:
                best_val_auc = val_auc
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    break

        # ============================
        # Final validation metrics
        # ============================
        val_metrics = {
            "acc": BinaryAccuracy()(preds_tensor, targs_tensor).item(),
            "sens": BinaryRecall()(preds_tensor, targs_tensor).item(),
            "spec": BinarySpecificity()(preds_tensor, targs_tensor).item(),
            "f1":   BinaryF1Score()(preds_tensor, targs_tensor).item(),
            "auc":  BinaryAUROC()(probs_tensor, targs_tensor).item(),
        }
        
        fold_metrics.append(val_metrics)

        aucs.append(val_metrics["auc"])

        del model
        torch.cuda.empty_cache()

    mean_metrics = {
        k: float(np.mean([fm[k] for fm in fold_metrics]))
        for k in fold_metrics[0]
    }

    for k, v in mean_metrics.items():
        trial.set_user_attr(k, v)
         

    return float(np.mean(aucs)) if aucs else 0.5
        

import os

print("=" * 60, flush=True)
print(f"STARTING JOB: REP={REP} FOLD={FOLD}", flush=True)
print(f"Device: {device}", flush=True)
print("=" * 60, flush=True)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))

# ==================================================
# Nested CV
# ==================================================
def run_nested_cv(XX, YY, sample_ids, confounds, n_reps, outer_splits=5):

    unique_ids = np.unique(sample_ids)
    unique_labels = np.array([YY[sample_ids == uid][0] for uid in unique_ids])

    all_metrics = []
    all_metrics_val = []
    all_ig = []
    all_deltas = []

    # Select the repetition
    rep = REP
    set_seed(42 + rep)


    outer = StratifiedKFold(n_splits=outer_splits, shuffle=True, random_state=1234 + rep)

    for fold, (train, test) in enumerate(outer.split(unique_ids, unique_labels)):

        
        # SELECT FOLD
        if fold != FOLD:
            continue
        log(f"Starting CV: REP={REP}, FOLD={FOLD}")

        train_ids = unique_ids[train]
        test_ids = unique_ids[test]

        train_mask = np.isin(sample_ids, train_ids)
        test_mask = np.isin(sample_ids, test_ids)

        X_train, y_train = XX[train_mask], YY[train_mask]
        X_test, y_test = XX[test_mask], YY[test_mask]

        C_train, C_test = confounds[train_mask], confounds[test_mask]

        # Optimize the hyperparameters in the inner 3-fold cv
        study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=2))

        # For logging
        log(f"Starting Optuna optimization for REP={rep}, FOLD={fold}")

        study.optimize(lambda t: objective(t, X_train, y_train, sample_ids[train_mask], C_train), n_trials=10)
        
        log(
            f"Optuna finished. "
            f"Best AUC={study.best_value:.4f}"
        )


        best = study.best_params
        
        log(
            f"Best parameters: "
            f"batch_size={best['batch_size']}, "
            f"lr={best['lr']:.2e}, "
            f"hidden_size={best['hidden_size']}, "
            f"num_layers={best['num_layers']}, "
            f"block_size={best['block_size']}"
        )

        best_trial = study.best_trial
        val_metrics = best_trial.user_attrs

        # Validation metrics
        for m, v in val_metrics.items():
            all_metrics_val.append({
                "rep": rep,
                "fold": fold,
                "level": "validation",
                "metric": m,
                "value": v
            })


        conf = ConfoundRegressor()
        X_train_corr, beta = conf.fit_transform(X_train, C_train)
        X_test_corr = X_test - (C_test @ beta)

        model = ResNet([best["hidden_size"]] * best["num_layers"], [best["block_size"]] * best["num_layers"]).to(device)

        opt = torch.optim.Adam(model.parameters(), lr=best["lr"], betas=(best["beta1"], 0.999), weight_decay=best["weight_decay"])

        loss_fn = nn.CrossEntropyLoss()

        trainloader = spectral_dataloader(X_train_corr, y_train, sample_ids[train_mask], batch_size=best["batch_size"])

        # Train the final model
        use_amp = device.type == "cuda"
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
        
        log("Starting final model training")

        for epoch in range(30):
            model.train()
            for xb, yb, _ in trainloader:
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)
                opt.zero_grad()
                with torch.cuda.amp.autocast(enabled=use_amp):
                    outputs = model(xb)
                    loss = loss_fn(outputs, yb)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
                
        log("Final model training finished")

        # Download the testing set
        testloader = spectral_dataloader(X_test_corr, y_test, sample_ids[test_mask], batch_size=best["batch_size"], shuffle=False)

        # Test the model
        results = test_model(model, testloader, device, loss_fn, threshold=0.5)
        log("Model testing finished")

        # -------- Classification metrics --------
        for level, key in [("patient", "sample"), ("spectral", "spectral")]:
            preds = results[f"{key}_preds"]
            targs = results[f"{key}_targets"]
            probs = results[f"{key}_probs"]

            metrics = {
                "acc": BinaryAccuracy()(preds, targs).item(),
                "sens": BinaryRecall()(preds, targs).item(),
                "spec": BinarySpecificity()(preds, targs).item(),
                "f1":   BinaryF1Score()(preds, targs).item(),
                "auc":  BinaryAUROC()(probs, targs).item(),
            }

            for m, v in metrics.items():
                all_metrics.append({
                    "rep": rep,
                    "fold": fold,
                    "level": level,
                    "metric": m,
                    "value": v
                })

        # -------- Integrated Gradients on test data --------
                
        log("Starting Integrated Gradients")

        ig_dict, delta_dict = compute_integrated_gradients(
            model, testloader, trainloader, device
        )

        for cls, d in delta_dict.items():
            all_deltas.append({
                "rep": rep,
                "fold": fold,
                "class": cls,
                "delta_mean": float(np.mean(d)),
                "delta_std": float(np.std(d))
            })

        n_wavenumbers = next(iter(ig_dict.values())).shape[-1]

        all_ig.append({
            "rep": rep,
            "fold": fold,
            "ig": {
                0: ig_dict[0].mean(axis=0) if 0 in ig_dict else np.zeros(n_wavenumbers),
                1: ig_dict[1].mean(axis=0) if 1 in ig_dict else np.zeros(n_wavenumbers),
            }
        })

        log("Integrated Gradients finished")

    return (
        pd.DataFrame(all_metrics),
        all_ig,
        pd.DataFrame(all_deltas),
        pd.DataFrame(all_metrics_val)
    )
    log(f"REP={REP}, FOLD={FOLD} finished successfully")

# ==================================================
# Run
# ==================================================
metrics_df, ig_df, delta_df, metrics_df_val = run_nested_cv(XX, YY, sample_id_vector, confound_matrix, n_reps=10)
# ==================================================
# Save
# ==================================================
outdir = Path("/scratch/project_XX/AnalysisFiles/ResNet")
outdir.mkdir(parents=True, exist_ok=True)

metrics_df.to_csv(outdir / f"metrics_rep{REP}_fold{FOLD}.csv", index=False)
joblib.dump(ig_df, outdir / f"ig_rep{REP}_fold{FOLD}.joblib")
delta_df.to_csv(outdir / f"delta_rep{REP}_fold{FOLD}.csv", index=False)
metrics_df_val.to_csv(outdir / f"val_metrics_rep{REP}_fold{FOLD}.csv", index=False)